# 06 — Statistical Analysis and Classification

Use the extracted EMIT spectra for:

1. one-way ANOVA at selected spectral regions
2. pairwise Welch's t-tests
3. PCA
4. Random Forest classification
5. SVM classification
6. cross-validated confusion matrices and accuracy metrics

The selected key wavelengths follow the existing project notebook:
450, 550, 675, 720, 800, 900, 1600, and 2000 nm.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from itertools import combinations
from scipy import stats

from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.metrics import (
    confusion_matrix, classification_report, accuracy_score
)

DATA_FILE = Path("../outputs/spectral_profiles/EMIT_extracted_spectra.csv")
OUT_DIR = Path("../outputs/tables")
OUT_FIG = Path("../outputs/figures")
OUT_DIR.mkdir(parents=True, exist_ok=True)
OUT_FIG.mkdir(parents=True, exist_ok=True)

CLASSES = ["Stress", "Moderate", "Healthy"]
KEY_REGIONS = [
    ("Blue", 450),
    ("Green", 550),
    ("Red", 675),
    ("Red-Edge", 720),
    ("NIR", 800),
    ("NIR Plateau", 900),
    ("SWIR-1", 1600),
    ("SWIR-2", 2000),
]

In [ ]:
df = pd.read_csv(DATA_FILE)
metadata_cols = ["Point_Index", "Class", "X", "Y"]
wavelength_cols = [c for c in df.columns if c not in metadata_cols]
wavelengths = np.array([float(c) for c in wavelength_cols])

X = df[wavelength_cols].to_numpy(dtype=float)
y = df["Class"].to_numpy()

# Replace non-positive values with column means.
X[X <= 0] = np.nan
col_means = np.nanmean(X, axis=0)
inds = np.where(np.isnan(X))
X[inds] = np.take(col_means, inds[1])

print("Samples:", X.shape[0])
print("Bands:", X.shape[1])
print("Classes:")
print(pd.Series(y).value_counts())

In [ ]:
def nearest_idx(wl, target):
    return int(np.argmin(np.abs(wl - target)))

def sig_label(p):
    if p < 0.001:
        return "***"
    if p < 0.01:
        return "**"
    if p < 0.05:
        return "*"
    return "ns"

anova_rows = []
ttest_rows = []

for region, target_wl in KEY_REGIONS:
    idx = nearest_idx(wavelengths, target_wl)

    groups = {
        cls: X[y == cls, idx]
        for cls in CLASSES
    }

    f_stat, p_anova = stats.f_oneway(*groups.values())

    anova_rows.append({
        "Region": region,
        "Wavelength_nm": wavelengths[idx],
        "F_statistic": f_stat,
        "p_value": p_anova,
        "Significance": sig_label(p_anova),
    })

    for c1, c2 in combinations(CLASSES, 2):
        t_stat, p_t = stats.ttest_ind(
            groups[c1], groups[c2], equal_var=False
        )
        ttest_rows.append({
            "Region": region,
            "Wavelength_nm": wavelengths[idx],
            "Pair": f"{c1} vs {c2}",
            "t_statistic": t_stat,
            "p_value": p_t,
            "Significance": sig_label(p_t),
        })

anova_df = pd.DataFrame(anova_rows)
ttest_df = pd.DataFrame(ttest_rows)

anova_df.to_csv(OUT_DIR / "ANOVA_results.csv", index=False)
ttest_df.to_csv(OUT_DIR / "Welch_ttest_results.csv", index=False)

display(anova_df)
display(ttest_df)

In [ ]:
# PCA
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

pca = PCA(n_components=3)
X_pca = pca.fit_transform(X_scaled)

explained = pca.explained_variance_ratio_ * 100
print("Explained variance (%):", explained)

fig, ax = plt.subplots(figsize=(8, 6))

for cls in CLASSES:
    mask = y == cls
    ax.scatter(
        X_pca[mask, 0],
        X_pca[mask, 1],
        s=45,
        alpha=0.8,
        label=cls,
    )

ax.set_xlabel(f"PC1 ({explained[0]:.2f}%)")
ax.set_ylabel(f"PC2 ({explained[1]:.2f}%)")
ax.set_title("PCA of EMIT Hyperspectral Samples")
ax.grid(alpha=0.25)
ax.legend()
plt.tight_layout()
fig.savefig(OUT_FIG / "PCA_PC1_PC2.png", dpi=300)
plt.show()

In [ ]:
# Cross-validated Random Forest and SVM
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

models = {
    "Random Forest": RandomForestClassifier(
        n_estimators=300,
        random_state=42,
        class_weight="balanced",
        n_jobs=-1,
    ),
    "SVM": SVC(
        kernel="rbf",
        C=1.0,
        gamma="scale",
    ),
}

summary = []

for name, model in models.items():
    pred = cross_val_predict(model, X_scaled, y, cv=cv)

    acc = accuracy_score(y, pred)
    cm = confusion_matrix(y, pred, labels=CLASSES)

    print(f"\n{name}")
    print("Accuracy:", round(acc, 4))
    print(classification_report(y, pred, labels=CLASSES, zero_division=0))
    print("Confusion matrix:")
    print(pd.DataFrame(cm, index=CLASSES, columns=CLASSES))

    summary.append({
        "Model": name,
        "Accuracy": acc,
    })

summary_df = pd.DataFrame(summary)
summary_df.to_csv(OUT_DIR / "classification_accuracy.csv", index=False)
display(summary_df)